In [107]:
from asyncio.windows_events import NULL

import requests
import pandas as pd
import kagglehub
import numpy as np

from kagglehub import KaggleDatasetAdapter
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [108]:
file_path = "clinical_pneumonia_dataset.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "ajithdari/pneumonia-prediction-dataset",
  file_path
)

print("First 5 records:\n", df.head())
df.info()

First 5 records:
   patient_id   timestamp  note_sequence  \
0   PNE-6733  2022-06-11              0   
1   PNE-6733  2022-06-12              1   
2   PNE-6733  2022-06-13              2   
3   PNE-6733  2022-06-14              3   
4   PNE-6733  2022-06-15              4   

                                                                   clinical_note  \
0                 Pyrexia. rapid heart rate. rales. hypoxemia. productive cough.   
1                       Fever. tachycardia. rales. spo2 < 94%. productive cough.   
2         Raised body temp. tachycardia. crackles. spo2 < 94%. productive cough.   
3              Rapid heart rate. rales. low oxygen saturation. productive cough.   
4  Febrile. rapid heart rate. crackles. low oxygen saturation. productive cough.   

   fever  tachycardia  crackles  oxygen_saturation  wbc_count  \
0      1            1         1          93.500000   7.500000   
1      1            1         1          93.500000   7.500000   
2      1            1  

In [109]:
# GET DATA
url = "https://srhdpeuwpubsa.blob.core.windows.net/whdh/COVID/WHO-COVID-19-global-daily-data.csv"
df = pd.read_csv(url)

response = requests.head(url)
ukuran = int(response.headers.get('content-length', 0))

ukuran_mb = ukuran / (1024 * 1024)
print(f"Ukuran File CSV di URL: {ukuran_mb:.2f} MB")

Ukuran File CSV di URL: 25.32 MB


In [110]:
# Cleaning null values kolom New_cases
df_clean = df.dropna(subset=['New_cases']).copy()

# Persiapan Fitur untuk Clustering

# Menggunakan fitur New_cases
X = df_clean[['New_cases']].values

# Normalisasi data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Menerapkan K-Means dengan 3 Cluster (Sesuai rekomendasi penelitian)
k_means = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['Cluster'] = k_means.fit_predict(X_scaled)

# 4. Mapping Label zona Berdasarkan Rata-rata Nilai Klaster
# Mengurutkan cluster ID berdasarkan pusat centroid dari terkecil ke terbesar
centroid_order = np.argsort(k_means.cluster_centers_.sum(axis=1))

# Pemetaan: Centroid terendah -> Rendah (Kuning), Sedang -> Sedang (Oranye), Tertinggi -> Tinggi (Merah)
cluster_mapping = {
    centroid_order[0]: 'Rendah (Zona Kuning)',
    centroid_order[1]: 'Sedang (Zona Oranye)',
    centroid_order[2]: 'Tinggi (Zona Merah)',
}

# Terapkan pemetaan ke dataframe
df_clean['Risk_Level'] = df_clean['Cluster'].map(cluster_mapping)

# (Opsional) Jika ada daerah dengan 0 kasus secara eksplisit, dapat dikategorikan "Zona Hijau"
df_clean.loc[df_clean['New_cases'] == 0, 'Risk_Level'] = (
    'Tidak Ada Kasus (Zona Hijau)'
)

# Hapus kolom helper sementara jika tidak diperlukan
df_clean.drop(columns=['Cluster'], inplace=True)

In [111]:
# 1. UNSUPERVISED LEARNING - K-Means
X_usl = df_clean[['Cumulative_cases', 'Cumulative_deaths']]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['Cluster_USL'] = kmeans.fit_predict(X_usl)

# 2. FEATURE ENGINEERING (Ekstraksi Waktu & Wilayah)
df_clean['Date_reported'] = pd.to_datetime(df_clean['Date_reported'])
df_clean['Month'] = df_clean['Date_reported'].dt.month
df_clean['DayOfWeek'] = df_clean['Date_reported'].dt.dayofweek

# One-Hot Encoding untuk wilayah WHO_region
df_encoded = pd.get_dummies(df_clean, columns=['WHO_region'], drop_first=True)

# 3. PREPARASI DATASET SUPERVISED LEARNING
features = ['Cumulative_cases', 'Cumulative_deaths', 'Cluster_USL', 'Month', 'DayOfWeek'] + \
           [col for col in df_encoded.columns if col.startswith('WHO_region_')]

X = df_encoded[features]
y = df_encoded['Risk_Level']

In [112]:
import sqlite3
import pandas as pd

# Data Growth Checking
data_growth = len(df_clean)

print(f"--- PENGECEKAN NODE 17 ---")
print(f"Jumlah baris data siap simpan: {data_growth} baris")

if data_growth > 0:
    print("[STATUS: YES] Pertumbuhan data > 0.\n")

    # STORE DATA BY PERIODIC (DATABASE)
    # Connect ke SQLite
    conn = sqlite3.connect('covid19_bigdata_storage.db')

    # Menyimpan DataFrame ke dalam tabel database 'covid19_periodic_data'
    df_clean.to_sql('covid19_periodic_data', conn, if_exists='replace', index=False)

    print("[STATUS: SUCCESS] Data berhasil disimpan ke Database 'covid19_bigdata_storage.db'")

    # Verifikasi Data
    query_check = "SELECT Date_reported, Country, New_cases, Cumulative_cases, Risk_Level, Cluster_USL FROM covid19_periodic_data LIMIT 5"
    df_from_db = pd.read_sql_query(query_check, conn)

    conn.close()

    print("\nPreview Baris Database SQL")
    print(df_from_db)

else:
    print("[STATUS: NO] Data kosong!")

--- PENGECEKAN NODE 17 ---
Jumlah baris data siap simpan: 215026 baris
[STATUS: YES] Pertumbuhan data > 0.

[STATUS: SUCCESS] Data berhasil disimpan ke Database 'covid19_bigdata_storage.db'

Preview Baris Database SQL
         Date_reported            Country  New_cases  Cumulative_cases  \
0  2020-01-04 00:00:00     American Samoa        0.0                 0   
1  2020-01-04 00:00:00             Bhutan        0.0                 0   
2  2020-01-04 00:00:00         Bangladesh        0.0                 0   
3  2020-01-04 00:00:00  Brunei Darussalam        0.0                 0   
4  2020-01-04 00:00:00              China        1.0                 1   

                     Risk_Level  Cluster_USL  
0  Tidak Ada Kasus (Zona Hijau)            0  
1  Tidak Ada Kasus (Zona Hijau)            0  
2  Tidak Ada Kasus (Zona Hijau)            0  
3  Tidak Ada Kasus (Zona Hijau)            0  
4          Rendah (Zona Kuning)            0  


In [113]:
# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Machine Learning (Random Forest - Pruned Hyperparameters)
model_sl = RandomForestClassifier(
    n_estimators=10, max_depth=2, random_state=42
)
model_sl.fit(X_train, y_train)

# CEK AKURASI MODEL
y_pred = model_sl.predict(X_test)
acc = accuracy_score(y_test, y_pred) * 100

print(f"--- Evaluasi Sesuai Flowchart ---")
print(f"Akurasi Model: {acc:.2f}%")

if 70 <= acc < 90:
    print(
        "[STATUS: PAS] Akurasi memenuhi syarat (70% <= x < 90%). Lanjut ke"
        " tahap berikutnya!"
    )
else:
    print(
        "[STATUS: REVISI] Akurasi di luar rentang. Sesuaikan hyperparameter"
        " model."
    )

print("\nDetail Laporan Klasifikasi:")
print(classification_report(y_test, y_pred, zero_division=0))

--- Evaluasi Sesuai Flowchart ---
Akurasi Model: 88.11%
[STATUS: PAS] Akurasi memenuhi syarat (70% <= x < 90%). Lanjut ke tahap berikutnya!

Detail Laporan Klasifikasi:
                              precision    recall  f1-score   support

        Rendah (Zona Kuning)       0.88      0.99      0.93     51923
        Sedang (Zona Oranye)       0.00      0.00      0.00        28
Tidak Ada Kasus (Zona Hijau)       0.94      0.42      0.58     12554
         Tinggi (Zona Merah)       0.00      0.00      0.00         3

                    accuracy                           0.88     64508
                   macro avg       0.45      0.35      0.38     64508
                weighted avg       0.89      0.88      0.86     64508



In [114]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

def kaggle_importing(source, file_name):
    print(f'Load Dataset from {source}...')
    dataframe = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        source, file_name)

    if dataframe == NULL:
        print('GAGAL MEMUAT DATASET KAGGLE')
        return NULL
    return dataframe